In [1]:
import os
import kagglehub

# Укажите желаемую папку для скачивания
os.environ["KAGGLEHUB_CACHE"] = "/home/fedosdan2/Study/EdgeChess-AI/CV/datasets"  # или "./data" для текущей папки

# Download latest version
path = kagglehub.dataset_download("mmkoya/chessrender360")

print("Path to dataset files:", path)

/home/fedosdan2/miniconda3/envs/myenv/lib/python3.10/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/fedosdan2/miniconda3/envs/myenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 24.1G/24.1G [52:52<00:00, 8.15MB/s]  

Extracting files...


Path to dataset files: /home/fedosdan2/Study/EdgeChess-AI/CV/datasets/datasets/mmkoya/chessrender360/versions/5


In [31]:
import os
import json
import shutil
import random
import cv2
from pathlib import Path

# ==========================================
# 1. НАСТРОЙКА ПУТЕЙ И ПАРАМЕТРОВ
# ==========================================
DATASET_ROOT = "/home/fedosdan2/Study/EdgeChess-AI/CV/datasets/datasets/mmkoya/chessrender360/versions/5/ChessRender360" 
JSON_DIR = os.path.join(DATASET_ROOT, "annotations") 
IMG_DIR = os.path.join(DATASET_ROOT, "rgb")          

OUTPUT_ROOT = "/home/fedosdan2/Study/EdgeChess-AI/CV/yolo_chess_dataset"
os.makedirs(f"{OUTPUT_ROOT}/images/train", exist_ok=True)
os.makedirs(f"{OUTPUT_ROOT}/images/val", exist_ok=True)
os.makedirs(f"{OUTPUT_ROOT}/labels/train", exist_ok=True)
os.makedirs(f"{OUTPUT_ROOT}/labels/val", exist_ok=True)

TARGET_SIZE = (1000, 1000) 
CORNER_BBOX_SIZE = 100   

# ==========================================
# 2. КАРТА КЛАССОВ (17 классов: 6 белых + 6 чёрных + доска + 4 угла)
# ==========================================
CLASS_MAPPING = {
    "white_pawn": 0,
    "white_rook": 1,
    "white_knight": 2,
    "white_bishop": 3,
    "white_queen": 4,
    "white_king": 5,
    "black_pawn": 6,
    "black_rook": 7,
    "black_knight": 8,
    "black_bishop": 9,
    "black_queen": 10,
    "black_king": 11,
    "chessboard": 12,
    "corner_white_left": 13,
    "corner_white_right": 14,
    "corner_black_left": 15,
    "corner_black_right": 16
}

# ==========================================
# 3. РАЗДЕЛЕНИЕ ДАННЫХ (80% train / 20% val)
# ==========================================
json_files = [f for f in os.listdir(JSON_DIR) if f.endswith('.json')]
random.seed(42)
random.shuffle(json_files)

split_idx = int(len(json_files) * 0.8)
train_files = json_files[:split_idx]
val_files = json_files[split_idx:]

print(f"✅ Найдено {len(json_files)} аннотаций.")
print(f"📊 Train: {len(train_files)} | Val: {len(val_files)}")

# ==========================================
# 4. ФУНКЦИЯ КОНВЕРТАЦИИ ОДНОГО ФАЙЛА
# ==========================================
def convert_to_yolo(json_filename, split_type):
    json_path = os.path.join(JSON_DIR, json_filename)
    
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    img_filename = os.path.basename(data.get("rgb_path", ""))
    if not img_filename: 
        return
        
    img_path = os.path.join(IMG_DIR, img_filename)
    if not os.path.exists(img_path): 
        return

    # --- Читаем и ресайзим картинку ---
    img = cv2.imread(img_path)
    if img is None: 
        return
    
    img_resized = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)
    
    dest_img_path = os.path.join(OUTPUT_ROOT, "images", split_type, img_filename)
    cv2.imwrite(dest_img_path, img_resized)

    new_h, new_w = TARGET_SIZE[1], TARGET_SIZE[0]
    
    scale_x = new_w / img.shape[1]
    scale_y = new_h / img.shape[0]

    yolo_lines = []

    # --- 4.1. ДОСКА (frame_bbox) ---
    if "frame_bbox" in data:
        x_min, y_min, x_max, y_max = data["frame_bbox"]
        x_min, x_max = x_min * scale_x, x_max * scale_x
        y_min, y_max = y_min * scale_y, y_max * scale_y
        
        x_center = ((x_min + x_max) / 2.0) / new_w
        y_center = ((y_min + y_max) / 2.0) / new_h
        width = (x_max - x_min) / new_w
        height = (y_max - y_min) / new_h
        
        yolo_lines.append(f"12 {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

    # --- 4.2. ФИГУРЫ (с разделением на белые/чёрные) ---
    if "pieces" in data:
        for piece in data["pieces"]:
            piece_name = piece.get("piece_name", "")  # например, "white_pawn"
            bbox = piece.get("bbox", [])
            if len(bbox) != 4: continue
            
            # Теперь используем полное имя (white_pawn, black_king и т.д.)
            if piece_name in CLASS_MAPPING:
                class_id = CLASS_MAPPING[piece_name]
                x_min, y_min, x_max, y_max = bbox
                
                x_min, x_max = x_min * scale_x, x_max * scale_x
                y_min, y_max = y_min * scale_y, y_max * scale_y
                
                x_center = ((x_min + x_max) / 2.0) / new_w
                y_center = ((y_min + y_max) / 2.0) / new_h
                width = (x_max - x_min) / new_w
                height = (y_max - y_min) / new_h
                
                yolo_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

    # --- 4.3. УГЛЫ ДОСКИ (board_corners) ---
    if "board_corners" in data:
        corners = data["board_corners"]
        corner_mapping = {
            "white_left": "corner_white_left",
            "white_right": "corner_white_right",
            "black_left": "corner_black_left",
            "black_right": "corner_black_right"
        }
        
        for json_key, class_name in corner_mapping.items():
            if json_key in corners:
                px, py = corners[json_key]
                
                px_scaled = px * scale_x
                py_scaled = py * scale_y
                
                half_size = (CORNER_BBOX_SIZE * scale_x) / 2.0
                
                x_min = px_scaled - half_size
                y_min = py_scaled - half_size
                x_max = px_scaled + half_size
                y_max = py_scaled + half_size
                
                x_min = max(0, x_min)
                y_min = max(0, y_min)
                x_max = min(new_w, x_max)
                y_max = min(new_h, y_max)
                
                x_center = ((x_min + x_max) / 2.0) / new_w
                y_center = ((y_min + y_max) / 2.0) / new_h
                width = (x_max - x_min) / new_w
                height = (y_max - y_min) / new_h
                
                class_id = CLASS_MAPPING[class_name]
                yolo_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

    # --- 4.4. СОХРАНЯЕМ .txt ФАЙЛ ---
    txt_filename = Path(img_filename).stem + ".txt"
    dest_txt_path = os.path.join(OUTPUT_ROOT, "labels", split_type, txt_filename)
    
    with open(dest_txt_path, "w", encoding="utf-8") as f:
        f.write("\n".join(yolo_lines))

# ==========================================
# 5. ЗАПУСК ОБРАБОТКИ
# ==========================================
print("🔄 Конвертация Train набора...")
for jf in train_files: 
    convert_to_yolo(jf, "train")

print("🔄 Конвертация Val набора...")
for jf in val_files: 
    convert_to_yolo(jf, "val")

print("✅ Конвертация изображений и аннотаций завершена!")

# ==========================================
# 6. ГЕНЕРАЦИЯ data.yaml
# ==========================================
abs_output_root = os.path.abspath(OUTPUT_ROOT)

yaml_content = f"""path: {abs_output_root}
train: images/train
val: images/val

nc: 17
names:
  0: white_pawn
  1: white_rook
  2: white_knight
  3: white_bishop
  4: white_queen
  5: white_king
  6: black_pawn
  7: black_rook
  8: black_knight
  9: black_bishop
  10: black_queen
  11: black_king
  12: chessboard
  13: corner_white_left
  14: corner_white_right
  15: corner_black_left
  16: corner_black_right

# Аугментации
hsv_h: 0.015
hsv_s: 0.7
hsv_v: 0.4
degrees: 10.0
translate: 0.1
scale: 0.5
perspective: 0.005
flipud: 0.0
fliplr: 0.5
mosaic: 0.5
mixup: 0.0
"""

yaml_path = os.path.join(OUTPUT_ROOT, "data.yaml")
with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(yaml_content)

print(f"📄 Файл конфигурации создан: {yaml_path}")
print(" Датасет полностью готов к обучению!")

✅ Найдено 10000 аннотаций.
📊 Train: 8000 | Val: 2000
🔄 Конвертация Train набора...
🔄 Конвертация Val набора...
✅ Конвертация изображений и аннотаций завершена!
📄 Файл конфигурации создан: /home/fedosdan2/Study/EdgeChess-AI/CV/yolo_chess_dataset/data.yaml
 Датасет полностью готов к обучению!


In [36]:
import cv2
import os
import random
import numpy as np
from pathlib import Path


OUTPUT_ROOT = "/home/fedosdan2/Study/EdgeChess-AI/CV/yolo_chess_dataset"
IMAGE_TRAIN = f"{OUTPUT_ROOT}/images/train"
LABEL_TRAIN = f"{OUTPUT_ROOT}/labels/train"

# ID -> имя класса (числовые ключи!)
CLASS_NAMES = {
    0: "white_pawn",
    1: "white_rook",
    2: "white_knight",
    3: "white_bishop",
    4: "white_queen",
    5: "white_king",
    6: "black_pawn",
    7: "black_rook",
    8: "black_knight",
    9: "black_bishop",
    10: "black_queen",
    11: "black_king",
    12: "chessboard",
    13: "corner_white_left",
    14: "corner_white_right",
    15: "corner_black_left",
    16: "corner_black_right"
}

COLORS = {
    0: (0, 200, 0),     # white_pawn - зелёный
    1: (0, 150, 0),     # white_rook
    2: (0, 255, 100),   # white_knight
    3: (100, 255, 100), # white_bishop
    4: (200, 255, 200), # white_queen
    5: (255, 255, 255), # white_king
    6: (0, 0, 200),     # black_pawn - синий
    7: (0, 0, 150),     # black_rook
    8: (100, 100, 255), # black_knight
    9: (100, 100, 200), # black_bishop
    10: (200, 200, 255),# black_queen
    11: (0, 0, 255),    # black_king
    12: (255, 255, 255),# chessboard - белый
    13: (255, 255, 0),  # corner_white_left - жёлтый
    14: (255, 128, 0),  # corner_white_right - оранжевый
    15: (128, 0, 255),  # corner_black_left - фиолетовый
    16: (0, 255, 128)   # corner_black_right - бирюзовый
}


def draw_yolo_bbox(image, label_file):
    """Читает YOLO .txt файл и рисует bounding box'ы на изображении."""
    img_height, img_width = image.shape[:2]
    corners = {}

    if not os.path.exists(label_file):
        print(f"⚠️ Файл разметки не найден: {label_file}")
        return image, None

    with open(label_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()

        for line in lines:
            parts = line.strip().split()
            if len(parts) != 5:
                continue

            class_id = int(parts[0])
            x_center = float(parts[1])
            y_center = float(parts[2])
            width = float(parts[3])
            height = float(parts[4])

            x_center_px = x_center * img_width
            y_center_px = y_center * img_height
            w_px = width * img_width
            h_px = height * img_height

            x_min = int(x_center_px - w_px / 2)
            y_min = int(y_center_px - h_px / 2)
            x_max = int(x_center_px + w_px / 2)
            y_max = int(y_center_px + h_px / 2)

            x_min = max(0, x_min)
            y_min = max(0, y_min)
            x_max = min(img_width, x_max)
            y_max = min(img_height, y_max)

            # Сохраняем координаты углов (ID 13-16)
            if class_id in [13, 14, 15, 16]:
                corners[class_id] = (int(x_center_px), int(y_center_px))
                
                cx = int(x_center_px)
                cy = int(y_center_px)
                color = COLORS.get(class_id, (128, 128, 128))
                cv2.drawMarker(image, (cx, cy), color, cv2.MARKER_CROSS, 20, 2)

            # chessboard = 12
            thickness = 3 if class_id == 12 else 2
            color = COLORS.get(class_id, (128, 128, 128))
            cv2.rectangle(image, (x_min, y_min), (x_max, y_max), color, thickness)

            label = CLASS_NAMES.get(class_id, f"class_{class_id}")
            (text_width, text_height), baseline = cv2.getTextSize(
                label, cv2.FONT_HERSHEY_SIMPLEX, 0.4, 1
            )
            cv2.rectangle(image, (x_min, y_min - text_height - baseline), (x_min + text_width, y_min), color, -1)
            cv2.putText(image, label, (x_min, y_min - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 0) if class_id == 12 else (255, 255, 255), 1)

    return image, corners


def get_ordered_corners_with_orientation(corners_dict, img_width, img_height):
    """
    Упрощённая версия: всегда разворачивает доску белыми вниз.
    """
    white_left = np.array(corners_dict[13])
    white_right = np.array(corners_dict[14])
    black_left = np.array(corners_dict[15])
    black_right = np.array(corners_dict[16])
    
    ordered = [black_right, black_left, white_left, white_right]
    white_at_bottom = 1
    
    return ordered, white_at_bottom


def apply_perspective_transform(image, corners, output_size=900, padding=50):
    """
    Применяет Perspective Transform по 4 углам доски с правильной ориентацией.
    """
    src_points = np.float32(corners)
    
    total_size = output_size + 2 * padding
    
    dst_points = np.float32([
        [output_size + padding, padding],
        [padding, padding],                      # top_left = a8
        [output_size + padding, output_size + padding],  # bottom_right = h1
        [padding, output_size + padding]         # bottom_left = a1
    ])
  
    M = cv2.getPerspectiveTransform(src_points, dst_points)
    warped = cv2.warpPerspective(image, M, (total_size, total_size))
    
    return warped, M


def draw_8x8_grid(image, output_size=900, padding=50, color=(0, 255, 255), thickness=1, white_at_bottom=True):
    """
    Рисует сетку 8x8 на изображении с учетом ориентации.
    """
    h, w = image.shape[:2]
    cell_size = output_size / 8
    
    # Вертикальные линии
    for i in range(1, 8):
        x = int(padding + i * cell_size)
        cv2.line(image, (x, padding), (x, padding + output_size), color, thickness)
    
    # Горизонтальные линии
    for i in range(1, 8):
        y = int(padding + i * cell_size)
        cv2.line(image, (padding, y), (padding + output_size, y), color, thickness)
    
    # Подписи колонок (a-h) и рядов (1-8)
    cols = "abcdefgh"
    for i in range(8):
        if white_at_bottom:
            # Буквы колонок снизу (ряд 1)
            cv2.putText(
                image, cols[i],
                (int(padding + i * cell_size + cell_size / 2 - 5), padding + output_size + 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1
            )
            # Цифры рядов слева (1-8 снизу вверх)
            cv2.putText(
                image, str(i + 1),
                (padding - 25, int(padding + (7 - i) * cell_size + cell_size / 2 + 5)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1
            )
        else:
            # Перевернутая ориентация: буквы сверху, ряды 8-1 сверху вниз
            cv2.putText(
                image, cols[len(cols) - i - 1],
                (int(padding + i * cell_size + cell_size / 2 - 5), padding - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1
            )
            cv2.putText(
                image, str(i+1),
                (padding - 25, int(padding + i * cell_size + cell_size / 2 + 5)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1
            )
    
    return image


num = 0
# ==========================================
# ЗАПУСК
# ==========================================
img_files = [f for f in os.listdir(IMAGE_TRAIN) if f.endswith(('.jpg', '.jpeg', '.png'))]

if not img_files:
    print(" Изображения в папке train не найдены! Проверь пути.")
else:
    img_name_no_ext = Path(f"{IMAGE_TRAIN}/rgb_{num}.jpeg").stem
    label_path = f"{LABEL_TRAIN}/rgb_{num}.txt"
     
    selected_img = Path(f"{IMAGE_TRAIN}/rgb_{num}.jpeg")
    img_path = f"{IMAGE_TRAIN}/rgb_{num}.jpeg"

    print(f"🖼️  Визуализация изображения: {selected_img}")
    print(f"📄  Файл разметки: {img_name_no_ext}.txt")

    img = cv2.imread(img_path)
    if img is None:
        print("❌ Не удалось прочитать изображение. Проверь путь.")
    else:
        img_height, img_width = img.shape[:2]

        img_annotated, corners = draw_yolo_bbox(img.copy(), label_path)

        required_corners = {13, 14, 15, 16}
        found_corners = set(corners.keys())
        missing_corners = required_corners - found_corners

        if missing_corners:
            print(f"⚠️ Не найдены углы: {[CLASS_NAMES[c] for c in missing_corners]}")
            output_path = f"visualized_{selected_img}"
            cv2.imwrite(output_path, img_annotated)
            print(f"✅ Сохранено: {output_path}")
        else:
            print(f"✅ Все 4 угла найдены:")
            for cid, (x, y) in corners.items():
                print(f"   {CLASS_NAMES[cid]}: ({x}, {y})")

            # Определяем порядок углов и ориентацию
            ordered_corners, white_at_bottom = get_ordered_corners_with_orientation(
                corners, img_width, img_height
            )
            
            print(f"📐 Ориентация: белая сторона {'внизу' if white_at_bottom else 'вверху'}")

            # Perspective Transform
            output_size = 900
            padding = 50
            warped_board, M = apply_perspective_transform(img, ordered_corners, output_size, padding)

            # Рисуем сетку
            warped_with_grid = draw_8x8_grid(
                warped_board.copy(), output_size, padding, 
                color=(0, 255, 255), thickness=1, white_at_bottom=white_at_bottom
            )

            # Переносим фигуры
            if os.path.exists(label_path):
                with open(label_path, 'r', encoding='utf-8') as f:
                    lines = f.readlines()

                for line in lines:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue

                    class_id = int(parts[0])
                    # Пропускаем доску (12) и углы (13-16)
                    if class_id in [12, 13, 14, 15, 16]:
                        continue

                    x_center = float(parts[1]) * img_width
                    y_center = float(parts[2]) * img_height
                    height = float(parts[4]) * img_height

                    # Сдвигаем точку от центра вниз на 40% высоты bounding box
                    piece_x = x_center
                    piece_y = y_center + (height * 0.4)

                    # Трансформируем именно эту смещенную точку
                    point = np.float32([[[piece_x, piece_y]]])
                    transformed_point = cv2.perspectiveTransform(point, M)
                    tx, ty = transformed_point[0][0]

                    cv2.circle(warped_with_grid, (int(tx), int(ty)), 5,
                               COLORS.get(class_id, (128, 128, 128)), -1)

                    # Определяем клетку
                    col = int((tx - padding) / (output_size / 8))
                    row = int((ty - padding) / (output_size / 8))
                    col = max(0, min(7, col))
                    row = max(0, min(7, row))

                    if white_at_bottom:
                        cell_name = f"{'abcdefgh'[col]}{8 - row}"
                    else:
                        cell_name = f"{'hgfedcba'[col]}{row + 1}"
                    
                    label = CLASS_NAMES.get(class_id, "?")

                    cv2.putText(
                        warped_with_grid, f"{label}({cell_name})",
                        (int(tx) + 8, int(ty) + 4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.35,
                        COLORS.get(class_id, (128, 128, 128)), 1
                    )

            # Собираем коллаж
            scale = 800 / img_height
            img_annotated_resized = cv2.resize(
                img_annotated,
                (int(img_width * scale), 800),
                interpolation=cv2.INTER_AREA
            )

            gap = 40
            collage_w = img_annotated_resized.shape[1] + gap + warped_board.shape[1]
            collage_h = max(800, warped_board.shape[0]) + 60

            collage = np.ones((collage_h, collage_w, 3), dtype=np.uint8) * 40

            cv2.putText(collage, "Original + BBox + Corners",
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
            cv2.putText(collage, f"Perspective Transform (white {'bottom' if white_at_bottom else 'top'})",
                        (img_annotated_resized.shape[1] + gap + 10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

            offset_x = img_annotated_resized.shape[1] + gap
            collage[60:60 + img_annotated_resized.shape[0], 0:img_annotated_resized.shape[1]] = img_annotated_resized
            collage[60:60 + warped_board.shape[0], offset_x:offset_x + warped_board.shape[1]] = warped_with_grid

            output_path = f"collage_corners.jpeg"
            cv2.imwrite(output_path, collage)
            print(f"✅ Коллаж сохранен: {output_path}")

            warped_path = f"warped_corners.jpeg"
            cv2.imwrite(warped_path, warped_with_grid)
            print(f"✅ Выпрямленная доска сохранена: {warped_path}")

🖼️  Визуализация изображения: /home/fedosdan2/Study/EdgeChess-AI/CV/yolo_chess_dataset/images/train/rgb_0.jpeg
📄  Файл разметки: rgb_0.txt
✅ Все 4 угла найдены:
   corner_white_left: (566, 249)
   corner_white_right: (173, 386)
   corner_black_left: (381, 772)
   corner_black_right: (877, 494)
📐 Ориентация: белая сторона внизу
✅ Коллаж сохранен: collage_corners.jpeg
✅ Выпрямленная доска сохранена: warped_corners.jpeg


In [ ]:
from ultralytics import YOLO
import os

# ==========================================
# ПРОСТОЕ ОБУЧЕНИЕ YOLO НА ПОЛНОМ ДАТАСЕТЕ
# ==========================================

# Путь к датасету
DATASET_PATH = "/home/fedosdan2/Study/EdgeChess-AI/CV/yolo_chess_dataset"

# Создаём data.yaml
data_yaml = f"""path: {DATASET_PATH}
train: images/train
val: images/val

nc: 17
names:
  0: white_pawn
  1: white_rook
  2: white_knight
  3: white_bishop
  4: white_queen
  5: white_king
  6: black_pawn
  7: black_rook
  8: black_knight
  9: black_bishop
  10: black_queen
  11: black_king
  12: chessboard
  13: corner_white_left
  14: corner_white_right
  15: corner_black_left
  16: corner_black_right

# Аугментации
hsv_h: 0.015
hsv_s: 0.7
hsv_v: 0.4
degrees: 10.0
translate: 0.1
scale: 0.5
perspective: 0.005
flipud: 0.0
fliplr: 0.5
mosaic: 0.5
mixup: 0.0
"""

with open("chess_data.yaml", "w") as f:
    f.write(data_yaml)

print("✅ Создан chess_data.yaml")

# Загружаем предобученную модель YOLOv8n
model = YOLO("yolov8n.pt")

# Запускаем обучение
print("🚀 Начинаем обучение...")

results = model.train(
    data="chess_data.yaml",
    epochs=100,           # Количество эпох
    imgsz=640,            # Размер изображения
    batch=8,              # Размер батча (уменьши до 4 если не хватает памяти)
    device=0,             # GPU (0 = первая видеокарта)
    name="chess_training",  # Имя папки с результатами
    patience=20,          # Ранняя остановка если нет улучшений 20 эпох
    save=True,            # Сохранять чекпоинты
    plots=True,           # Строить графики
    cache=False,          # Не кэшировать в RAM
    workers=4,            # Потоки загрузки данных
    optimizer='AdamW',    # Оптимизатор
    verbose=True,         # Подробный вывод
)

print("✅ Обучение завершено!")
print(f"📊 Лучшие веса: runs/detect/chess_training/weights/best.pt")

# Валидация
print("\n📈 Валидация...")
metrics = model.val(data="chess_data.yaml")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

# Метрики по классам
print("\n📊 Метрики по классам:")
for i, class_name in enumerate(model.names.values()):
    print(f"  {class_name}: mAP50 = {metrics.box.ap[i]:.4f}")

✅ Создан chess_data.yaml
🚀 Начинаем обучение...
Ultralytics 8.4.137 🚀 Python-3.10.21 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 5773MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=chess_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic